In [0]:
%pip install langchain-databricks langchain-core langgraph xgboost scikit-learn tiktoken

In [0]:
dbutils.library.restartPython()

In [0]:

import numpy as np
import json
import time
import tiktoken
from sklearn.datasets import fetch_covtype
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, log_loss
from xgboost import XGBClassifier
from langchain_databricks import ChatDatabricks
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage, ToolMessage
from langchain_core.tools import tool

SEED = 42
MAX_ITERATIONS = 15
NUM_RUNS = 3
SEEDS = [42, 123, 456]
MODEL_ENDPOINT = "databricks-claude-haiku-4-5"

_enc = tiktoken.get_encoding("cl100k_base")
def count_tokens(text: str) -> int:
    return len(_enc.encode(str(text)))

# ---------------------------------------------------------------------------
# Dataset: Covertype (Forest Cover Type) — UCI / sklearn built-in
# 61,878 samples, 93 features, 9 classes. Kaggle competition.
# Baseline XGBoost ~65% macro F1, good solutions ~80%+
# ---------------------------------------------------------------------------
def load_dataset(seed=SEED):
    """Load Covertype (forest cover type). 7 classes, 54 features, 581K samples.
    Subsample to 20K for speed. Imbalanced: class 4 has <3% of samples."""
    X_full, y_full = fetch_covtype(return_X_y=True)
    y_full = y_full - 1  # 0-indexed
    X, _, y, _ = train_test_split(X_full, y_full, train_size=20000, random_state=seed, stratify=y_full)
    n_classes = len(np.unique(y))
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=seed, stratify=y)
    print(f"  Dataset: {len(X):,} samples, {X.shape[1]} features, {n_classes} classes")
    print(f"  Class distribution: {dict(zip(*np.unique(y, return_counts=True)))}")
    print(f"  Train: {len(X_train):,}, Test: {len(X_test):,}")
    return X_train, X_test, y_train, y_test, n_classes


def train_and_evaluate(params: dict, X_train, X_test, y_train, y_test, n_classes) -> dict:
    """Train XGBoost multiclass, return macro F1 + logloss."""
    safe_params = {
        "n_estimators": int(params.get("n_estimators", 100)),
        "max_depth": int(params.get("max_depth", 6)),
        "learning_rate": float(params.get("learning_rate", 0.3)),
        "subsample": float(params.get("subsample", 1.0)),
        "colsample_bytree": float(params.get("colsample_bytree", 1.0)),
        "min_child_weight": int(params.get("min_child_weight", 1)),
        "gamma": float(params.get("gamma", 0.0)),
        "reg_alpha": float(params.get("reg_alpha", 0.0)),
        "reg_lambda": float(params.get("reg_lambda", 1.0)),
        "num_class": n_classes,
        "objective": "multi:softprob",
        "eval_metric": "mlogloss",
        "random_state": SEED,
        "n_jobs": -1,
        "tree_method": "hist",
    }
    t0 = time.time()
    model = XGBClassifier(**safe_params)
    model.fit(X_train, y_train, verbose=False)
    train_time = time.time() - t0

    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)
    macro_f1 = round(f1_score(y_test, y_pred, average='macro'), 4)
    ll = round(log_loss(y_test, y_proba), 4)
    return {
        "macro_f1": macro_f1,
        "logloss": ll,
        "train_time": round(train_time, 1),
        "params": safe_params,
    }

DEFAULT_PARAMS = {
    "n_estimators": 100, "max_depth": 6, "learning_rate": 0.3,
    "subsample": 1.0, "colsample_bytree": 1.0, "min_child_weight": 1,
    "gamma": 0.0, "reg_alpha": 0.0, "reg_lambda": 1.0,
}

def config_key(params):
    kp = {k: round(float(v), 3) if isinstance(v, (int, float)) else v
          for k, v in sorted(params.items())
          if k in ["n_estimators","max_depth","learning_rate","subsample",
                    "colsample_bytree","min_child_weight","gamma","reg_alpha","reg_lambda"]}
    return json.dumps(kp)

def extract_json(text):
    text = str(text).strip()
    if "```" in text:
        parts = text.split("```")
        for p in parts:
            if "{" in p:
                text = p.strip()
                if text.startswith("json"): text = text[4:].strip()
                break
    start = text.find("{")
    end = text.rfind("}") + 1
    if start >= 0 and end > start:
        return json.loads(text[start:end])
    raise ValueError(f"No JSON found")

SYSTEM_PROMPT = """You are an ML experiment agent optimizing XGBoost for the Covertype forest cover type classification task.

Dataset: 20,000 samples (subsampled from 581K), 54 features (10 quantitative + 44 binary), 7 cover type classes.
Target metric: macro F1 score (higher is better). Secondary: logloss (lower is better).

The dataset is imbalanced — class 4 (Cottonwood/Willow) has <3% of samples while class 1 (Spruce/Fir) and class 2 (Lodgepole Pine) dominate.
With default XGBoost hyperparameters, macro F1 is typically around 0.65-0.70. Good solutions reach 0.80+.

You can change:
- n_estimators (100-1000): more trees = more capacity but slower
- max_depth (3-12): deeper trees capture interactions but risk overfitting
- learning_rate (0.01-0.3): lower = more trees needed, but often better generalization
- subsample (0.5-1.0): row sampling per tree
- colsample_bytree (0.3-1.0): feature sampling per tree
- min_child_weight (1-20): minimum instance weight in a leaf
- gamma (0.0-5.0): minimum loss reduction for split
- reg_alpha (0.0-5.0): L1 regularization
- reg_lambda (0.5-5.0): L2 regularization

Key challenges:
- 7-class classification with significant class imbalance
- Mix of continuous and binary features
- Lower learning rate + more trees generally helps
- Regularization is important to prevent overfitting

Rules:
1. ONE experiment per call. Return valid JSON with hyperparameters.
2. Think about WHY a change might help before proposing it.
3. Don't repeat failed configurations.
4. The gap between baseline (~0.65) and good (~0.80) is wide — be strategic."""

print("Shared code loaded.")

In [0]:

def run_stateless(run_seed):
    """Karpathy-style: full history in prompt each iteration."""
    llm = ChatDatabricks(endpoint=MODEL_ENDPOINT, max_tokens=512)
    X_train, X_test, y_train, y_test, n_classes = load_dataset(run_seed)

    baseline = train_and_evaluate(DEFAULT_PARAMS, X_train, X_test, y_train, y_test, n_classes)
    history = [{"iteration": 0, "params": DEFAULT_PARAMS,
                "macro_f1": baseline["macro_f1"], "logloss": baseline["logloss"],
                "train_time": baseline["train_time"]}]
    best_f1 = baseline["macro_f1"]
    best_params = DEFAULT_PARAMS.copy()
    total_input_tokens = 0
    total_output_tokens = 0
    redundant_count = 0
    tried_configs = set()
    tried_configs.add(config_key(DEFAULT_PARAMS))
    llm_calls = 0
    start_time = time.time()

    for iteration in range(1, MAX_ITERATIONS + 1):
        # Full history reconstruction each time
        history_text = "FULL EXPERIMENT HISTORY:\n"
        for h in history:
            history_text += (f"  Iter {h['iteration']}: macro_F1={h['macro_f1']}, "
                           f"logloss={h['logloss']}, train_time={h['train_time']}s, "
                           f"params={json.dumps({k:v for k,v in h['params'].items() if k not in ['num_class','objective','eval_metric','random_state','n_jobs','tree_method']})}\n")
        best_iter = max(history, key=lambda h: h["macro_f1"])["iteration"]
        history_text += f"\nBest macro F1: {best_f1} (iteration {best_iter})\n"
        history_text += f"Experiments run: {len(history)}, Budget remaining: {MAX_ITERATIONS - iteration + 1}\n"
        history_text += f"\nPropose the next hyperparameter configuration. Return ONLY JSON."

        input_tokens = count_tokens(SYSTEM_PROMPT) + count_tokens(history_text)
        total_input_tokens += input_tokens

        response = llm.invoke([SystemMessage(content=SYSTEM_PROMPT),
                               HumanMessage(content=history_text)])
        llm_calls += 1
        total_output_tokens += count_tokens(response.content)

        try:
            params = extract_json(response.content)
        except:
            params = DEFAULT_PARAMS.copy()

        ck = config_key(params)
        if ck in tried_configs: redundant_count += 1
        tried_configs.add(ck)

        try:
            result = train_and_evaluate(params, X_train, X_test, y_train, y_test, n_classes)
        except Exception as e:
            result = {"macro_f1": 0.0, "logloss": 99.0, "train_time": 0, "params": params}

        history.append({"iteration": iteration, "params": params,
                        "macro_f1": result["macro_f1"], "logloss": result["logloss"],
                        "train_time": result["train_time"]})
        if result["macro_f1"] > best_f1:
            best_f1 = result["macro_f1"]
            best_params = params.copy()

        print(f"    Iter {iteration}: F1={result['macro_f1']:.4f} (best={best_f1:.4f}) [{result['train_time']}s]")

    return {
        "method": "stateless", "run_seed": run_seed, "history": history,
        "best_f1": best_f1, "best_params": best_params,
        "total_input_tokens": total_input_tokens,
        "total_output_tokens": total_output_tokens,
        "total_tokens": total_input_tokens + total_output_tokens,
        "redundant_count": redundant_count,
        "redundancy_rate": redundant_count / MAX_ITERATIONS,
        "wall_clock_seconds": time.time() - start_time,
        "llm_calls": llm_calls, "tool_calls": 0,
    }

print("Stateless runner loaded.")

In [0]:

from typing import TypedDict, Annotated, Literal
from langgraph.graph import StateGraph, END
from langgraph.prebuilt import ToolNode

_tool_call_counter = {"count": 0}
_run_state = {
    "X_train": None, "X_test": None, "y_train": None, "y_test": None,
    "n_classes": 0, "history": [], "best_f1": 0.0, "best_iteration": -1,
    "tried_configs": set(), "iteration": 0, "redundant_count": 0,
}

@tool
def get_experiment_history() -> str:
    """Get the full history of all experiments run so far, including hyperparameters, macro F1, and logloss."""
    _tool_call_counter["count"] += 1
    if not _run_state["history"]:
        return "No experiments run yet."
    lines = []
    for h in _run_state["history"]:
        p = {k:v for k,v in h['params'].items() if k not in ['num_class','objective','eval_metric','random_state','n_jobs','tree_method']}
        lines.append(f"Iter {h['iteration']}: F1={h['macro_f1']}, logloss={h['logloss']}, "
                     f"time={h['train_time']}s, params={json.dumps(p)}")
    lines.append(f"\nBest F1: {_run_state['best_f1']} (iter {_run_state['best_iteration']})")
    lines.append(f"Experiments: {len(_run_state['history'])}, Budget: {MAX_ITERATIONS - _run_state['iteration']}")
    return "\n".join(lines)

@tool
def run_experiment(params_json: str) -> str:
    """Train XGBoost with the given hyperparameters and return macro F1, logloss, and training time.

    Args:
        params_json: JSON string with hyperparameters, e.g. {"n_estimators": 300, "max_depth": 8, "learning_rate": 0.05, ...}
    """
    _tool_call_counter["count"] += 1
    try:
        params = json.loads(params_json)
    except:
        return "ERROR: Invalid JSON."

    _run_state["iteration"] += 1
    iteration = _run_state["iteration"]

    if iteration > MAX_ITERATIONS:
        return f"Budget exhausted ({MAX_ITERATIONS} iterations). Stop experimenting."

    ck = config_key(params)
    if ck in _run_state["tried_configs"]:
        _run_state["redundant_count"] += 1
    _run_state["tried_configs"].add(ck)

    try:
        result = train_and_evaluate(
            params, _run_state["X_train"], _run_state["X_test"],
            _run_state["y_train"], _run_state["y_test"], _run_state["n_classes"])
    except Exception as e:
        return f"ERROR: Training failed: {str(e)}"

    _run_state["history"].append({
        "iteration": iteration, "params": result["params"],
        "macro_f1": result["macro_f1"], "logloss": result["logloss"],
        "train_time": result["train_time"]
    })

    improved = result["macro_f1"] > _run_state["best_f1"]
    if improved:
        _run_state["best_f1"] = result["macro_f1"]
        _run_state["best_iteration"] = iteration

    status = "NEW BEST!" if improved else f"No improvement. Best={_run_state['best_f1']} (iter {_run_state['best_iteration']})"
    remaining = MAX_ITERATIONS - iteration
    return (f"Iter {iteration}: macro_F1={result['macro_f1']}, logloss={result['logloss']}, "
            f"train_time={result['train_time']}s. {status}. {remaining} iterations remaining.")

REACT_TOOLS = [get_experiment_history, run_experiment]

class ReactState(TypedDict):
    messages: Annotated[list, lambda a, b: a + b]

def should_continue(state: ReactState) -> Literal["tools", "end"]:
    last = state["messages"][-1]
    if hasattr(last, "tool_calls") and last.tool_calls:
        return "tools"
    return "end"

def agent_node(state: ReactState) -> dict:
    llm = ChatDatabricks(endpoint=MODEL_ENDPOINT, max_tokens=1024)
    llm_with_tools = llm.bind_tools(REACT_TOOLS)
    response = llm_with_tools.invoke(state["messages"])
    return {"messages": [response]}

def build_react_graph():
    graph = StateGraph(ReactState)
    graph.add_node("agent", agent_node)
    graph.add_node("tools", ToolNode(REACT_TOOLS))
    graph.set_entry_point("agent")
    graph.add_conditional_edges("agent", should_continue, {"tools": "tools", "end": END})
    graph.add_edge("tools", "agent")
    return graph.compile()

def run_stateful(run_seed):
    """LangGraph ReAct agent with tools."""
    X_train, X_test, y_train, y_test, n_classes = load_dataset(run_seed)

    _run_state.update({
        "X_train": X_train, "X_test": X_test,
        "y_train": y_train, "y_test": y_test,
        "n_classes": n_classes,
        "history": [], "best_f1": 0.0, "best_iteration": -1,
        "tried_configs": set(), "iteration": 0, "redundant_count": 0,
    })
    _tool_call_counter["count"] = 0

    baseline = train_and_evaluate(DEFAULT_PARAMS, X_train, X_test, y_train, y_test, n_classes)
    _run_state["history"].append({
        "iteration": 0, "params": baseline["params"],
        "macro_f1": baseline["macro_f1"], "logloss": baseline["logloss"],
        "train_time": baseline["train_time"]
    })
    _run_state["best_f1"] = baseline["macro_f1"]
    _run_state["best_iteration"] = 0
    _run_state["tried_configs"].add(config_key(DEFAULT_PARAMS))

    graph = build_react_graph()
    start_time = time.time()

    initial_msg = (
        f"You are optimizing XGBoost for Covertype classification (7 classes, 54 features, imbalanced). "
        f"Baseline (default params): macro_F1={baseline['macro_f1']}, logloss={baseline['logloss']}, "
        f"train_time={baseline['train_time']}s. "
        f"You have {MAX_ITERATIONS} experiments. Target: macro F1 > 0.80. "
        f"Use run_experiment to try new hyperparameters. Use get_experiment_history to review past results. "
        f"Be strategic — lower learning rate + more trees is a good starting direction."
    )

    state = {"messages": [SystemMessage(content=SYSTEM_PROMPT), HumanMessage(content=initial_msg)]}

    try:
        final_state = graph.invoke(state, {"recursion_limit": MAX_ITERATIONS * 6})
    except Exception as e:
        print(f"  Graph stopped: {e}")
        final_state = state

    elapsed = time.time() - start_time

    total_input_tokens = 0
    total_output_tokens = 0
    llm_calls = 0
    all_msgs = final_state.get("messages", state["messages"])
    for m in all_msgs:
        tokens = count_tokens(m.content if hasattr(m, 'content') else str(m))
        if isinstance(m, (HumanMessage, SystemMessage, ToolMessage)):
            total_input_tokens += tokens
        elif isinstance(m, AIMessage):
            total_output_tokens += tokens
            llm_calls += 1

    best_params = DEFAULT_PARAMS.copy()
    if _run_state["history"]:
        best_entry = max(_run_state["history"], key=lambda h: h["macro_f1"])
        best_params = {k:v for k,v in best_entry["params"].items()
                       if k not in ['num_class','objective','eval_metric','random_state','n_jobs','tree_method']}

    return {
        "method": "stateful", "run_seed": run_seed,
        "history": _run_state["history"],
        "best_f1": _run_state["best_f1"], "best_params": best_params,
        "total_input_tokens": total_input_tokens,
        "total_output_tokens": total_output_tokens,
        "total_tokens": total_input_tokens + total_output_tokens,
        "redundant_count": _run_state["redundant_count"],
        "redundancy_rate": _run_state["redundant_count"] / max(1, _run_state["iteration"]),
        "wall_clock_seconds": elapsed,
        "llm_calls": llm_calls, "tool_calls": _tool_call_counter["count"],
    }

print("Stateful ReAct runner loaded.")

In [0]:

all_results = []

print("=" * 70)
print("AUTORESEARCH BENCHMARK v3: Covertype Forest Cover Classification")
print(f"9 classes, 93 features, {MAX_ITERATIONS} iterations, {NUM_RUNS} seeds")
print("=" * 70)

for seed in SEEDS[:NUM_RUNS]:
    print(f"\n{'='*70}\nSEED {seed}\n{'='*70}")

    print(f"\n--- Stateless / Karpathy (seed={seed}) ---")
    r = run_stateless(run_seed=seed)
    print(f"  RESULT: F1={r['best_f1']:.4f}  Tokens={r['total_tokens']:,}  "
          f"Redundancy={r['redundancy_rate']:.0%}  Time={r['wall_clock_seconds']:.0f}s")
    all_results.append(r)

    print(f"\n--- Stateful / ReAct (seed={seed}) ---")
    r = run_stateful(run_seed=seed)
    print(f"  RESULT: F1={r['best_f1']:.4f}  LLM={r['llm_calls']}  Tools={r['tool_calls']}  "
          f"Tokens={r['total_tokens']:,}  Redundancy={r['redundancy_rate']:.0%}  Time={r['wall_clock_seconds']:.0f}s")
    all_results.append(r)

In [0]:

print("\n" + "=" * 70)
print("AGGREGATE RESULTS")
print("=" * 70)

for method in ["stateless", "stateful"]:
    runs = [r for r in all_results if r["method"] == method]
    print(f"\n{method.upper()} ({'Karpathy' if method == 'stateless' else 'LangGraph ReAct'}):")
    print(f"  Best F1:       {np.mean([r['best_f1'] for r in runs]):.4f} +/- {np.std([r['best_f1'] for r in runs]):.4f}")
    print(f"  Input tokens:  {np.mean([r['total_input_tokens'] for r in runs]):,.0f} +/- {np.std([r['total_input_tokens'] for r in runs]):,.0f}")
    print(f"  Output tokens: {np.mean([r['total_output_tokens'] for r in runs]):,.0f} +/- {np.std([r['total_output_tokens'] for r in runs]):,.0f}")
    print(f"  Total tokens:  {np.mean([r['total_tokens'] for r in runs]):,.0f} +/- {np.std([r['total_tokens'] for r in runs]):,.0f}")
    print(f"  LLM calls:     {np.mean([r['llm_calls'] for r in runs]):.0f}")
    print(f"  Tool calls:    {np.mean([r['tool_calls'] for r in runs]):.0f}")
    print(f"  Redundancy:    {np.mean([r['redundancy_rate'] for r in runs]):.1%}")
    print(f"  Wall-clock:    {np.mean([r['wall_clock_seconds'] for r in runs]):.0f}s +/- {np.std([r['wall_clock_seconds'] for r in runs]):.0f}s")

In [0]:

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import base64, io

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. Convergence curve (cumulative best F1)
ax = axes[0, 0]
for method, color, label in [("stateless", "#ef4444", "Stateless (Karpathy)"),
                              ("stateful", "#2563eb", "Stateful (ReAct)")]:
    runs = [r for r in all_results if r["method"] == method]
    curves = []
    for r in runs:
        best_so_far = []
        running_best = 0
        for h in r["history"]:
            running_best = max(running_best, h["macro_f1"])
            best_so_far.append(running_best)
        while len(best_so_far) < MAX_ITERATIONS + 1:
            best_so_far.append(best_so_far[-1] if best_so_far else 0)
        curves.append(best_so_far[:MAX_ITERATIONS + 1])
    curves = np.array(curves)
    mean_curve = curves.mean(axis=0)
    iters = range(len(mean_curve))
    ax.plot(iters, mean_curve, color=color, linewidth=2.5, label=label)
ax.set_xlabel("Experiment number", fontsize=11)
ax.set_ylabel("Best macro F1 (cumulative)", fontsize=11)
ax.set_title("Convergence", fontsize=12, fontweight='bold')
ax.legend(fontsize=10); ax.grid(True, alpha=0.3)
ax.set_ylim([0.55, 0.85])

# 2. Token consumption (stacked bars)
ax = axes[0, 1]
width = 0.35
for i, (method, label) in enumerate([("stateless", "Stateless"), ("stateful", "Stateful")]):
    runs = [r for r in all_results if r["method"] == method]
    inp = np.mean([r["total_input_tokens"] for r in runs])
    out = np.mean([r["total_output_tokens"] for r in runs])
    ax.bar(i, inp, width=0.5, label="Input" if i == 0 else "", color="#dbeafe", edgecolor="#2563eb", linewidth=1.5)
    ax.bar(i, out, width=0.5, bottom=inp, label="Output" if i == 0 else "", color="#fee2e2", edgecolor="#ef4444", linewidth=1.5)
    ax.text(i, inp + out + 200, f"{int(inp+out):,}", ha='center', fontsize=10, fontweight='bold')
ax.set_xticks([0, 1]); ax.set_xticklabels(["Stateless\n(Karpathy)", "Stateful\n(ReAct)"], fontsize=10)
ax.set_ylabel("Total tokens", fontsize=11)
ax.set_title("Token Consumption", fontsize=12, fontweight='bold')
ax.legend(fontsize=9); ax.grid(True, alpha=0.3, axis="y")

# 3. Per-iteration input tokens (showing scaling)
ax = axes[1, 0]
for method, color, label in [("stateless", "#ef4444", "Stateless"),
                              ("stateful", "#2563eb", "Stateful")]:
    runs = [r for r in all_results if r["method"] == method]
    if method == "stateless":
        # Estimate: system_prompt + history grows linearly
        tokens_per_iter = [count_tokens(SYSTEM_PROMPT) + 120 * i for i in range(1, MAX_ITERATIONS + 1)]
        ax.plot(range(1, MAX_ITERATIONS + 1), tokens_per_iter, color=color, linewidth=2.5, label=label)
    else:
        # Estimate: bounded at ~200 tokens per tool call
        tokens_per_iter = [200] * MAX_ITERATIONS
        ax.plot(range(1, MAX_ITERATIONS + 1), tokens_per_iter, color=color, linewidth=2.5, label=label)
ax.set_xlabel("Iteration", fontsize=11)
ax.set_ylabel("Input tokens per iteration", fontsize=11)
ax.set_title("Per-Iteration Token Cost", fontsize=12, fontweight='bold')
ax.legend(fontsize=10); ax.grid(True, alpha=0.3)
ax.annotate("O(n)", xy=(14, count_tokens(SYSTEM_PROMPT) + 120*14), fontsize=10, color="#ef4444", fontweight='bold')
ax.annotate("O(1)", xy=(14, 220), fontsize=10, color="#2563eb", fontweight='bold')

# 4. Wall-clock time
ax = axes[1, 1]
for i, (method, color) in enumerate([("stateless", "#fca5a5"), ("stateful", "#93c5fd")]):
    runs = [r for r in all_results if r["method"] == method]
    times = [r["wall_clock_seconds"] for r in runs]
    ax.bar(i, np.mean(times), width=0.5, yerr=np.std(times), capsize=5,
           color=color, edgecolor=["#ef4444", "#2563eb"][i], linewidth=1.5)
    ax.text(i, np.mean(times) + np.std(times) + 5, f"{np.mean(times):.0f}s", ha='center', fontsize=10, fontweight='bold')
ax.set_xticks([0, 1]); ax.set_xticklabels(["Stateless\n(Karpathy)", "Stateful\n(ReAct)"], fontsize=10)
ax.set_ylabel("Wall-clock time (seconds)", fontsize=11)
ax.set_title("Execution Time", fontsize=12, fontweight='bold')
ax.grid(True, alpha=0.3, axis="y")

plt.suptitle("AutoResearch Benchmark: Covertype Classification (7 classes, 54 features, 15 iterations)",
             fontsize=13, fontweight="bold", y=1.01)
plt.tight_layout()

buf = io.BytesIO()
fig.savefig(buf, format='png', dpi=150, bbox_inches='tight')
buf.seek(0)
img_b64 = base64.b64encode(buf.read()).decode('utf-8')
plt.close(fig)
displayHTML(f'<img src="data:image/png;base64,{img_b64}" />')

In [0]:

import pandas as pd

rows = []
for r in all_results:
    for h in r["history"]:
        rows.append({
            "method": r["method"], "run_seed": r["run_seed"],
            "iteration": h["iteration"], "macro_f1": h["macro_f1"],
            "logloss": h["logloss"], "train_time": h["train_time"],
            "params": json.dumps({k:v for k,v in h["params"].items()
                                  if k not in ['num_class','objective','eval_metric','random_state','n_jobs','tree_method']}),
        })
spark.createDataFrame(pd.DataFrame(rows)).write.mode("overwrite").option(
    "overwriteSchema", "true"
).saveAsTable("gc_prod_sandbox_mlproduct.tmp.autoresearch_benchmark_v3_iterations")

summary_rows = []
for r in all_results:
    summary_rows.append({
        "method": r["method"], "run_seed": r["run_seed"],
        "best_f1": r["best_f1"], "total_input_tokens": r["total_input_tokens"],
        "total_output_tokens": r["total_output_tokens"],
        "total_tokens": r["total_tokens"],
        "llm_calls": r["llm_calls"], "tool_calls": r["tool_calls"],
        "redundancy_rate": r["redundancy_rate"],
        "wall_clock_seconds": r["wall_clock_seconds"],
        "best_params": json.dumps(r["best_params"]),
    })
spark.createDataFrame(pd.DataFrame(summary_rows)).write.mode("overwrite").option(
    "overwriteSchema", "true"
).saveAsTable("gc_prod_sandbox_mlproduct.tmp.autoresearch_benchmark_v3_summary")

print("Results saved to Delta tables (v3).")